# 08 – Model Calibration and Probability Scaling

**Purpose:** Calibrate our fine-tuned `DistilBERT` sequence classifier probabilities using Temperature Scaling, and evaluate ECE/Brier scores.

This notebook demonstrates:
1. Loading validation and test logits.
2. Optimising the Temperature parameter $T$ on validation logits.
3. Applying scaling and comparing uncalibrated vs. calibrated probabilities.
4. Calculating Expected Calibration Error (ECE) and Brier scores.
5. Displaying Reliability Diagrams, Confidence Histograms, and Threshold routing curves.

## 0. Setup and Environment

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import IPython.display as display

REPO_ROOT = Path().resolve().parent
if REPO_ROOT.name != "SupportAI" and (REPO_ROOT / "SupportAI").exists():
    REPO_ROOT = REPO_ROOT / "SupportAI"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")

## 1. Execute Calibration Pipeline

We run the calibration runner to calculate the optimized temperature and export metrics.

In [ ]:
from src.evaluation.calibration import CalibrationRunner
from src.utils.constants import OUTPUT_DIR

model_dir = OUTPUT_DIR / "models" / "best_model"

runner = CalibrationRunner(model_dir, smoke_run=True)
metrics = runner.calibrate()

print("Calibration complete!")
print(f"Optimised Temperature: {metrics['optimized_temperature']:.4f}")
print(f"ECE (Before): {metrics['ece_before']:.4f} | ECE (After): {metrics['ece_after']:.4f}")
print(f"Brier (Before): {metrics['brier_before']:.4f} | Brier (After): {metrics['brier_after']:.4f}")

## 2. Display Reliability Diagrams and Histograms

Let's display the before and after diagnostic plots.

In [ ]:
plots_dir = OUTPUT_DIR / "metrics" / "plots"

print("Reliability Diagram (Confidence vs Accuracy Bins):")
display.display(display.Image(filename=str(plots_dir / "reliability_diagram.png")))

print("\nPrediction Confidences Distribution Density:")
display.display(display.Image(filename=str(plots_dir / "confidence_histogram.png")))

print("\nThreshold Routing Study:")
display.display(display.Image(filename=str(plots_dir / "threshold_study.png")))

In [ ]:
# Export Phase Manifest
import json
from src.utils.artifacts import save_yaml
from src.api.app import get_git_commit

calibration_metrics_path = REPO_ROOT / "outputs" / "metrics" / "calibration_metrics.json"
calibration_metrics = {}
if calibration_metrics_path.exists():
    with open(calibration_metrics_path) as f:
        calibration_metrics = json.load(f)

manifest = {
    "phase": "08_Calibration",
    "calibration_metrics": calibration_metrics,
    "git_commit": get_git_commit(),
}
save_yaml(manifest, REPO_ROOT / "outputs" / "manifests" / "phase_08_calibration.yaml")
print("YAML manifest saved successfully:")
print(manifest)
